# ResNet50 CPU Benchmark
Measures inference speed and power on the KV260 **ARM CPU (Cortex-A53)**.

**Run from any terminal on the KV260** (SSH or Jupyter terminal)

Uses ONNX Runtime with CPUExecutionProvider.

> ⚠️ CPU runs hot — keep N_BENCHMARK low (20). Let board cool between runs.

In [ ]:
import os, time, numpy as np

POWER_SENSOR = "/sys/class/hwmon/hwmon2/power1_input"
N_WARMUP = 3
N_BENCHMARK = 20

# Smart path: check local models/ first, then /home/ubuntu/
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("cpu_bench.ipynb"))
MODEL_LOCAL  = os.path.join(NOTEBOOK_DIR, "models", "resnet50-v1-7.onnx")
MODEL_UBUNTU = "/home/ubuntu/resnet50-v1-7.onnx"
MODEL_PATH   = MODEL_LOCAL if os.path.exists(MODEL_LOCAL) else MODEL_UBUNTU

def read_power_mw():
    with open(POWER_SENSOR) as f:
        return int(f.read().strip()) / 1000

print(f"Using model: {MODEL_PATH}")
print(f"Model exists: {os.path.exists(MODEL_PATH)}")
print(f"Current board power: {read_power_mw()/1000:.2f} W (idle)")

## Step 1 — Load ResNet50 ONNX Model
Model file: `/home/ubuntu/resnet50-v1-7.onnx` (98MB, pre-downloaded)

In [ ]:
import onnxruntime as ort
print(f"ONNX Runtime version: {ort.__version__}")

session = ort.InferenceSession(MODEL_PATH, providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
print(f"Input: {input_name} {session.get_inputs()[0].shape}")

## Step 2 — Prepare Input
ResNet50 expects `(1, 3, 224, 224)` — batch of 1, RGB image, 224x224 pixels.

In [ ]:
dummy_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
print(f"Input shape: {dummy_input.shape}")

## Step 3 — Warmup

In [ ]:
print(f"Warming up ({N_WARMUP} frames)...")
for _ in range(N_WARMUP):
    session.run(None, {input_name: dummy_input})
print("Warmup done!")

## Step 4 — Benchmark
Only 20 frames to avoid overheating the ARM CPU.

In [ ]:
print(f"Benchmarking ({N_BENCHMARK} frames)...")
power_samples = []
start = time.time()
for i in range(N_BENCHMARK):
    session.run(None, {input_name: dummy_input})
    if i % 5 == 0:
        power_samples.append(read_power_mw())
elapsed = time.time() - start
print("Done!")

## Step 5 — Results

In [ ]:
avg_power_w = sum(power_samples) / len(power_samples) / 1000
fps = N_BENCHMARK / elapsed
latency_ms = elapsed / N_BENCHMARK * 1000
fps_per_watt = fps / avg_power_w

print("=" * 40)
print("RESNET50 CPU BENCHMARK RESULTS")
print("=" * 40)
print(f"Platform:    KV260 ARM CPU (Cortex-A53)")
print(f"FPS:         {fps:.2f}")
print(f"Latency:     {latency_ms:.1f} ms/frame")
print(f"Power:       {avg_power_w:.2f} W")
print(f"FPS/Watt:    {fps_per_watt:.2f}")
print("=" * 40)